In [1]:
import math
import numpy as np

def prom_update(prom_prev: float, x_new: float, n_curr: int):
    if n_curr < 1:
        raise Exception()
    elif n_curr == 1:
        return x_new
    else:
        return prom_prev + (x_new - prom_prev) / n_curr

def s_cuad_update(s_cuad_prev: float, prom_prev: float, prom_curr: float, n_curr: int):
    if n_curr < 1:
        raise Exception()
    elif n_curr == 1:
        return 0.0
    else:
        return (1 - (1/(n_curr - 1))) * s_cuad_prev + n_curr * ((prom_curr - prom_prev)**2)


### Ejercicio 8.
Para el Ejercicio 7 suponer que:
- El tiempo de servicio del servidor se distribuye de manera exponencial, con una tasa de $13$ servicios por hora.
- Siempre que el servidor completa un trabajo y no encuentra trabajos para realizar, toma un descanso por un tiempo uniformemente distribuido en el intervalo $(0, 0.3)$ horas. A excepción de la primera solicitud
que es atendida de inmediato.
- Si al retomar no encuentra trabajos para realizar, vuelve a tomarse un descanso con la misma distribución.

Se pide:

a. Desarrollar un programa que simule el proceso durante un tiempo $T = 100$ horas, registrando los tiempos de llegada, los tiempos de servicio, los períodos de descanso del servidor y la evolución del número de trabajos en cola.

b. Utilizar el programa desarrollado en a) para estimar el tiempo promedio que tarda en ser procesada una solicitud. Detener las simulaciones cuando la desviación estándar muestral del estimador de la media sea menor que $0.05$ horas.

c. Estimar el tiempo medio que el servidor permanece en descanso. Detener las simulaciones cuando la desviación estándar muestral del estimador de la media sea menor que $0.05$ horas.

In [2]:
def lamda_t(t: float):
    t_m = t % 10

    if t_m <= 5:
        return 3*t_m + 4
    else:
        return -3*t_m + 34

def t_nueva_solicitud():
    lamda_cota = 19
    t = 0

    while True:
        t -= math.log(1 - np.random.random()) / lamda_cota
        V = np.random.random()

        if V < lamda_t(t) / lamda_cota:
            return t

def t_servicio():
    # Exponencial con lambda = 13 (13 servicios por hora)
    return -math.log(1 - np.random.random()) / 13

def t_reposo():
    # Uniforme({0, 0.3})
    return np.random.random() * 0.3

def punto_a(T: int = 100):
    # Registros
    A = [] # Tiempo en la que llega una solicitud
    D = [] # Tiempo en el cual se procesa una solicitud
    R = [] # Intervalo (t0, t1) donde t0 es cuando comienza el reposo y t1 cuando termina
    N = [0] # Cola de solicitudes
    N_T = [0] # Tiempo en el cual se cambia la cola de solicitudes

    # Inicialización
    t, NA, ND = 0, 0, 0
    n = 0

    tA = t + t_nueva_solicitud()
    tD = math.inf

    tR = math.inf
    inicio_reposo = 0

    while tA < math.inf or tD < math.inf:
        evento_prox = min(tA, tD, tR)

        # Llega nuevo solicitud
        if evento_prox == tA:
            t = tA
            NA += 1
            n += 1

            A.append(t)
            N.append(n)
            N_T.append(t)

            tA = t + t_nueva_solicitud()
            if tA > T:
                # El tiempo de la nueva solicitud supero T, por lo que no se
                # reciben nuevas solicitudes y solo se procesa las solicitudes
                # en la cola
                tA = math.inf

                # print(
                #    f"Ya pasaron las {T} horas, por lo que ya no llegan mas",
                #    f"solicitudes. Quedan {n} clientes por atender.\n")

            if n == 1 and tR == math.inf:
                tD = t + t_servicio()
            elif n == 1 and tR < math.inf:
                tD = t + (tR - inicio_reposo) + t_servicio()

        elif evento_prox == tD:
            # Se termina de procesar una solicitud
            t = tD
            ND += 1
            n -= 1

            D.append(t)
            N.append(n)
            N_T.append(t)

            if n > 0:
                # Se añade el siguiente evento de atención del servidor
                tD = t + t_servicio()
            else:
                # Se vacio la cola, por lo que, el servidor entra en reposo
                tD = math.inf
                tR = t + t_reposo()
                inicio_reposo = t

        else:
            # Termina tiempo de reposo
            t = tR

            R.append((inicio_reposo, t))

            if n > 0:
                # Hay clientes esperando, termina el descanso y empieza a atender
                tR = math.inf
                tD = t + t_servicio()
            else:
                # Sigue vacío, se toma otro descanso inmediatamente
                tR = t + t_reposo()
                inicio_reposo = t

    return t, A, D, R, N, N_T

# t, A, D, R, N, _ = punto_a()
# print(f"Simulación finalizada a las {t:.2f} horas")
# print(f"Total de llegadas registradas: {len(A)}")
# print(f"Total de servicios completados: {len(D)}")
# print(f"Total de descansos tomados: {len(R)}")
# print()

# print("Tiempos de llegada (A):", A[:10], "...")
# print("Tiempos de salida (D):", D[:10], "...")
# print("Intervalos de descanso (R):", R[:5], "...")
# print("Evolución de la cola (N):", N[:10], "...")

In [3]:
# Punto B
def generar_media_i():
    _, A_i, D_i, _, _, _ = punto_a()

    return (np.array(D_i) - np.array(A_i)).mean()

def punto_b():
    media = generar_media_i()
    s_cuad, n = 0, 1

    while n < 100 or math.sqrt(s_cuad / n) >= 0.05:
        X = generar_media_i()
        n += 1

        media_prev = media
        media = prom_update(media, X, n)
        s_cuad = s_cuad_update(s_cuad, media_prev, media, n)

    print(f"Cantidad de datos generados: ", n)
    print(f"Tiempo promedio que tarda una solicitud en ser procesada ~ {media:.4f} horas")
    print(f"Desviación estándar muestral = {math.sqrt(s_cuad / n):.4f} ")

punto_b()

Cantidad de datos generados:  100
Tiempo promedio que tarda una solicitud en ser procesada ~ 0.2133 horas
Desviación estándar muestral = 0.0013 


In [4]:
# Punto C
def generar_media_i():
    _, _, _, R, _, _ = punto_a()

    media = 0
    for i in range(len(R)):
        media += R[i][1] - R[i][0]

    media = media / len(R)
    return media

def punto_c():
    media = generar_media_i()
    s_cuad, n = 0, 1

    while n < 100 or math.sqrt(s_cuad / n) >= 0.05:
        X = generar_media_i()
        n += 1

        media_prev = media
        media = prom_update(media, X, n)
        s_cuad = s_cuad_update(s_cuad, media_prev, media, n)

    print(f"Cantidad de datos generados: ", n)
    print(f"Tiempo medio que el servidor permanece en descanzo ~ {media:.4f} horas")
    print(f"Desviación estándar muestral = {math.sqrt(s_cuad / n):.4f} ")

punto_c()

Cantidad de datos generados:  100
Tiempo medio que el servidor permanece en descanzo ~ 0.1507 horas
Desviación estándar muestral = 0.0003 
